# DeepTuneIO — Module 05: SLURM `seff` Metrics Extractor

This notebook documents and validates **Module 05** of the DeepTuneIO pipeline.

Module 05 runs **off-cluster**. It reads the JobID metadata CSV produced by Module 03,
locates the raw `seff` reports collected by Module 04, extracts SLURM resource metrics,
and writes an enriched CSV.

The notebook now uses the same campaign hierarchy adopted by DeepTuneIO-Indexer:

`CORA_RDR_UAB / Article / Data / Application / Dataset group / Scenario`


## 1. Project and module paths

The executable implementation is stored in `Module/05_extract_seff_metrics.py`.
The notebook does not duplicate route-building logic: it imports and uses the module.


In [1]:
from pathlib import Path
import sys
import pandas as pd

MODULE_DIR = Path("./Module").resolve()

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from importlib.util import spec_from_file_location, module_from_spec

MODULE05_FILE = MODULE_DIR / "05_extract_seff_metrics.py"
spec = spec_from_file_location("deeptuneio_module05", MODULE05_FILE)
module05 = module_from_spec(spec)
sys.modules[spec.name] = module05
spec.loader.exec_module(module05)

print("Module directory :", MODULE_DIR)
print("Module 05        :", MODULE05_FILE)
print("Exists           :", MODULE05_FILE.exists())
print("Version          :", module05.VERSION)


Module directory : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\03_Resource_Metadata\Module
Module 05        : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\03_Resource_Metadata\Module\05_extract_seff_metrics.py
Exists           : True
Version          : 2.0.0


## 2. Select application and campaign

Only the scientific campaign is selected here. The remaining paths are derived
automatically by Module 05.

The definitions below cover the configurations currently reconstructed for Article 01.


In [30]:
# ============================================================
# Selección de aplicación, dataset/campaña y escenario
# ============================================================

CAMPAIGNS = {
    "DeepGalaxy": {  # 1
        "DG_bw512_f3c5x64": [ # 1
            "lustre_1ost",   # 1
            "lustre_2ost",   # 2
            "lustre_4ost",   # 3 
            "nfs_SnGPU_e1",  # 4 
        ],
    },

    "DLIOv1": {  # 2
        "HDF5_64bs": [  # 1
            "lustre_ost" # 1
        ],
        "NPZ_64bs": [ # 2
            "lustre_ost"  # 1
        ],
        "TFRecord_256kts_64bs": [ # 3
            "lustre_ost"  # 1
        ],
        "TFRecord_1mts_64bs": [ # 4
            "lustre_ost"  # 1
        ],
    },
}


# ============================================================
# 1. Seleccionar aplicación
# ============================================================

APPLICATION_OPTIONS = list(CAMPAIGNS.keys())

APPLICATION_POS = 2
# 1 = DeepGalaxy
# 2 = DLIOv1

if not 1 <= APPLICATION_POS <= len(APPLICATION_OPTIONS):
    raise ValueError(
        f"APPLICATION_POS debe estar entre 1 y "
        f"{len(APPLICATION_OPTIONS)}"
    )

APPLICATION = APPLICATION_OPTIONS[APPLICATION_POS - 1]


# ============================================================
# 2. Seleccionar dataset/campaña
# ============================================================

DATASET_OPTIONS = list(CAMPAIGNS[APPLICATION].keys())

DATASET_POS = 4

if not 1 <= DATASET_POS <= len(DATASET_OPTIONS):
    raise ValueError(
        f"DATASET_POS debe estar entre 1 y "
        f"{len(DATASET_OPTIONS)}"
    )

DATASET_GROUP = DATASET_OPTIONS[DATASET_POS - 1]


# ============================================================
# 3. Seleccionar escenario
# ============================================================

SCENARIO_OPTIONS = CAMPAIGNS[APPLICATION][DATASET_GROUP]

SCENARIO_POS = 1

if not 1 <= SCENARIO_POS <= len(SCENARIO_OPTIONS):
    raise ValueError(
        f"SCENARIO_POS debe estar entre 1 y "
        f"{len(SCENARIO_OPTIONS)}"
    )

SCENARIO = SCENARIO_OPTIONS[SCENARIO_POS - 1]


# ============================================================
# Mostrar selección
# ============================================================

print("Selección actual")
print("----------------")
print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)


Selección actual
----------------
Application   : DLIOv1
Dataset group : TFRecord_1mts_64bs
Scenario      : lustre_ost


## 3. CORA root and campaign route

For the final CORA package, set `CORA_ROOT` to the root of `CORA_RDR_UAB`.
During local validation it can point to a local copy of that hierarchy.

The expected campaign structure is:

```text
<campaign_root>/
├── raw/
│   └── seff/
└── results/
    └── Resource_Metadata/
        ├── 03_jobids_from_parser_....csv
        └── 05_seff_metrics_....csv
```


In [31]:
# Adapt this root to the machine where Module 05 is executed.
from pathlib import Path

PROJECT_ROOT = Path(
    r"F:\NoteBooks\CORA_DR_UAB"
).resolve()

ARTICLE_ID = "Article_01_Parallel_IO_Analysis"

ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID

CAMPAIGN_ROOT = (
    ARTICLE_ROOT
    / "Data"
    / APPLICATION
    / DATASET_GROUP
    / SCENARIO
)

print("Project root  :", PROJECT_ROOT)
print("Article root  :", ARTICLE_ROOT)
print("Campaign root :", CAMPAIGN_ROOT)
print("Exists        :", CAMPAIGN_ROOT.exists())

Project root  : F:\NoteBooks\CORA_DR_UAB
Article root  : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\lustre_ost
Exists        : True


## 4. Resolve Module 05 inputs and outputs

`resolve_campaign_paths()` is the single source of truth for the Module 05 paths.
The notebook therefore does not manually rebuild `INPUT_CSV`, `SEFF_DIR` or
`OUTPUT_DIR`.


In [32]:
args = module05.parse_args([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-label", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", "JSC24",
])

try:
    paths = module05.resolve_campaign_paths(args)
    for key, value in paths.items():
        print(f"{key:14s}: {value}")
except Exception as exc:
    print("Routes cannot yet be fully resolved:", exc)
    print("This is expected if Module 03 output has not yet been generated.")


Routes cannot yet be fully resolved: No Module-03 JobID metadata CSV could be discovered. Expected under F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Metadata or F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results.
This is expected if Module 03 output has not yet been generated.


## 5. Run Module 05

Run this cell only after:

1. Module 03 has generated the JobID metadata CSV.
2. Module 04 has generated/copied the `seff_<jobid>.txt` reports into `raw/seff/`.

No HPC resources are used by this step.


In [34]:
return_code = module05.main([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-label", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", "JSC24",
])

print("Return code:", return_code)


DeepTuneIO — Module 05: SLURM seff Metrics Extractor
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost
Application   : DLIOv1
Scenario      : lustre_ost
Dataset label : TFRecord_1mts_64bs
Input CSV     : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Metadata\03_jobids_from_parser_DLIOv1_TFRecord_1mts_64bs_lustre_ost_JSC24.csv
JobID column  : jobid
seff dir      : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\raw\seff
seff reports  : 13
Rows          : 13
Output dir    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Metadata

Summary
  OK         : 13
  Incomplete : 0
  Missing    : 0
  Errors     : 0

CSV saved to: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Met

## 6. Inspect the generated metrics

Module 05 preserves the Module 03 metadata and appends the normalized `seff`
resource metrics.


In [ ]:
RESOURCE_METADATA_DIR = CAMPAIGN_ROOT / "results" / "Resource_Metadata"

generated = sorted(
    RESOURCE_METADATA_DIR.glob("05_seff_metrics_*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if generated:
    SEFF_METRICS_CSV = generated[0]
    seff_df = pd.read_csv(SEFF_METRICS_CSV)
    print("CSV:", SEFF_METRICS_CSV)
    display(seff_df)
else:
    print("No Module 05 output has been generated yet.")


## 7. Quality-control summary

The `seff_status` field distinguishes successful extraction from missing,
incomplete, or erroneous reports. Missing reports remain visible instead of
silently removing the corresponding experiment.


In [ ]:
if "seff_df" in globals() and not seff_df.empty:
    display(
        seff_df["seff_status"]
        .value_counts(dropna=False)
        .rename_axis("seff_status")
        .reset_index(name="executions")
    )

    problem_rows = seff_df[
        seff_df["seff_status"].fillna("missing") != "ok"
    ]

    if not problem_rows.empty:
        cols = [
            c for c in [
                "jobid", "JobID", "application", "dataset_label",
                "scenario", "seff_status", "seff_message", "seff_file"
            ]
            if c in problem_rows.columns
        ]
        display(problem_rows[cols])


## 8. Relationship with the DeepTuneIO pipeline

```text
processed/Parser
      │
      ▼
Module 03 — local
      │
      ├── JobID metadata CSV
      └── JobID manifest
                 │
                 ▼
Module 04 — HPC login/service node
                 │
                 └── raw/seff/seff_<jobid>.txt
                              │
                              ▼
Module 05 — local
                              │
                              └── results/Resource_Metadata/
                                  05_seff_metrics_....csv
```

Only Module 04 depends on the SLURM environment. Module 05 performs all parsing
and metric extraction outside the HPC system.
